# M18K V2 — Dataset Audit, EDA, and YOLOv8n-seg Reproduction

**Study:** Zakeri et al., *Computers* 14(5), 199 (2025), DOI: [10.3390/computers14050199](https://doi.org/10.3390/computers14050199).  
**Official project:** <https://github.com/abdollahzakeri/m18k>  

## Research/evaluation contract

- **Paper:** 2,000 main RGB-D images, 70% / 10% / 20% training / validation / unseen test. Table 2 uses F1 / AP / AR; YOLOv8-N *segmentation* AP = **0.712** at 640 and **0.846** at 1280. RGB-D Mask R-CNN did **not** improve over RGB in that paper.
- **Our default:** newer **V2 archive (repository says 4,652 images)**, use *existing split* when available; otherwise generate a **reproducible** random image split using seed 42 (not paper-identical and vulnerable to near-duplicate/session leakage). Test is only touched for final evaluation.
- **Mask model:** `yolov8n-seg.pt`; **quick pilot:** 10 epochs, 640 pixels. The paper-trained model used a longer unspecified training horizon and different compute. Expect differences.
- **Metrics:** compare COCO-like overall *mask AP@0.50:0.95* to paper AP only as an **informal reference** if annotation conventions and dataset/split differ. `mAP@50` is **not** equivalent to paper AP.
- **Training outputs:** runtime-generated data/weights are **not** committed to Git. Save result JSON, plots, notebook and EDA summary to Git.

**Prerequisite:** Colab (`Runtime → Change runtime type → T4 GPU`, if offered). Free GPU access is not guaranteed. This notebook can complete EDA without a GPU, but intentionally skips training on CPU unless you explicitly override.

In [ ]:
from pathlib import Path
import os, sys, shutil, json, random, hashlib, math, time, platform, subprocess
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
DATA_ROOT = ROOT / 'M18K_V2_data'                  # existing V2 dataset folder, or download destination
ARCHIVE = ROOT / 'M18KV2.tar.zst'
WORK_ROOT = ROOT / 'M18K_V2_reproduction'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
USE_OFFICIAL_V2_DOWNLOAD = True                   # set False if V2 is already in DATA_ROOT
RUN_TRAIN = True                                  # requires GPU by default
ALLOW_CPU_TRAIN = False                           # CPU would be VERY slow
EPOCHS = 10                                       # quick pilot; raise for full experiment
IMAGE_SIZE = 640                                  # later repeat at 1280 for a separate run
BATCH = 8
V2_URL = 'https://zakerimushroom.sfo3.digitaloceanspaces.com/M18KV2.tar.zst'

try:
    import torch
    HAS_GPU = bool(torch.cuda.is_available())
    print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'none')
except ImportError:
    HAS_GPU = False
    print('PyTorch not installed yet')
print('Available disk:', round(shutil.disk_usage(ROOT).free / (1024**3), 2), 'GiB')
print('Data path:', DATA_ROOT)

### 1 — Install dependencies

In [ ]:
%pip -q install "ultralytics>=8.3,<9" pycocotools opencv-python-headless pandas matplotlib pyyaml pillow

### 2 — Download and unpack the official V2 archive


In [ ]:
from google.colab import drive
from pathlib import Path
from tqdm.auto import tqdm
import requests
import shutil
import subprocess
import time
import os
import re

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Persistent paths
DRIVE_ROOT = Path('/content/drive/MyDrive/M18K_Reproduction')
DATA_ROOT = DRIVE_ROOT / 'M18K_V2_data' / 'M18KV2'
ARCHIVE = DRIVE_ROOT / 'M18KV2.tar.zst'
PARTIAL = DRIVE_ROOT / 'M18KV2.tar.zst.part'
READY_MARKER = DRIVE_ROOT / 'v2_extraction_complete.txt'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Official dataset URL
V2_URL = (
    'https://zakerimushroom.sfo3.digitaloceanspaces.com/'
    'M18KV2.tar.zst'
)

# 3. Find COCO annotation files
def find_coco_jsons(root):
    files = []

    if not root.exists():
        return files

    for p in root.rglob('*.json'):
        try:
            if p.stat().st_size == 0:
                continue

            with p.open('r', encoding='utf-8') as f:
                prefix = f.read(4000)

            name = p.name.lower()

            if ('"images"' in prefix and '"annotations"' in prefix):
                files.append(p)
            elif any(x in name for x in (
                'annotation', 'coco', 'instances'
            )):
                files.append(p)

        except (OSError, UnicodeError):
            continue

    return sorted(set(files))


def dataset_present():
    """Basic check for the extracted dataset."""
    json_files = find_coco_jsons(DATA_ROOT)

    if not json_files:
        return False

    # Find at least one RGB image
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        if next(DATA_ROOT.rglob(ext), None) is not None:
            return True

    return False


# 4. Resumable download
def download_with_resume(url, destination, partial, retries=6):

    if destination.exists():
        print("Archive already saved on Drive.")
        print(destination)
        return

    for attempt in range(1, retries + 1):

        downloaded = partial.stat().st_size if partial.exists() else 0

        headers = {}
        if downloaded:
            headers['Range'] = f'bytes={downloaded}-'
            print(f"Resuming from {downloaded / 2**30:.2f} GiB")

        try:
            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(40, 180)
            ) as response:

                if response.status_code in (401, 403, 404):
                    raise RuntimeError(
                        f"Dataset server returned HTTP {response.status_code}. "
                        "Use the manual-download fallback described below."
                    )

                response.raise_for_status()

                # If the server ignores Range, restart safely.
                resuming = (
                    downloaded > 0 and response.status_code == 206
                )

                if downloaded and not resuming:
                    print("Server did not support resume; restarting download.")
                    downloaded = 0

                expected_total = None

                if resuming:
                    content_range = response.headers.get(
                        'Content-Range', ''
                    )
                    match = re.match(
                        r'bytes (\d+)-\d+/(\d+|\*)',
                        content_range
                    )

                    if match:
                        if int(match.group(1)) != downloaded:
                            raise RuntimeError(
                                "Unexpected resume offset from server."
                            )
                        if match.group(2) != '*':
                            expected_total = int(match.group(2))

                content_length = int(
                    response.headers.get('Content-Length', 0)
                )

                if expected_total is None and content_length:
                    expected_total = downloaded + content_length

                mode = 'ab' if resuming else 'wb'

                with open(partial, mode) as output:
                    with tqdm(
                        total=expected_total,
                        initial=downloaded,
                        unit='B',
                        unit_scale=True,
                        desc='M18K V2 download'
                    ) as progress:

                        for chunk in response.iter_content(
                            chunk_size=4 * 1024 * 1024
                        ):
                            if chunk:
                                output.write(chunk)
                                progress.update(len(chunk))

                actual_size = partial.stat().st_size

                if expected_total and actual_size != expected_total:
                    raise IOError(
                        f"Incomplete download: {actual_size} "
                        f"of {expected_total} bytes"
                    )

            # Completed HTTP transfer
            os.replace(partial, destination)

            print("Download complete!")
            print("Archive:", destination)
            print("Size:", round(
                destination.stat().st_size / 2**30, 2
            ), "GiB")
            return

        except RuntimeError:
            raise

        except (requests.RequestException, OSError) as error:
            print(f"Attempt {attempt}/{retries} failed:", error)

            if attempt == retries:
                raise RuntimeError(
                    "Download interrupted repeatedly. "
                    f"Partial data is saved at: {partial}"
                ) from error

            time.sleep(min(5 * attempt, 20))


# 5. Main logic
if dataset_present():

    print("=" * 55)
    print("M18K V2 ALREADY AVAILABLE!")
    print("Skipping download and extraction.")
    print("=" * 55)

else:

    print("Checking Google Drive for M18K V2...")

    # Accept an archive manually downloaded with a .zip name
    manual_zip = DRIVE_ROOT / 'M18KV2.zip'

    if not ARCHIVE.exists() and manual_zip.exists():
        print("Found manually downloaded archive.")
        manual_zip.rename(ARCHIVE)

    # Download only if archive isn't already saved
    if not ARCHIVE.exists():
        print("Downloading official M18K V2 to Google Drive...")

        download_with_resume(
            V2_URL,
            ARCHIVE,
            PARTIAL
        )

    else:
        print("Reusing archive:", ARCHIVE)

    # Install zstd if missing
    if shutil.which('zstd') is None:
        print("Installing zstd...")

        subprocess.run(
            ['apt-get', '-qq', 'update'],
            check=True
        )

        subprocess.run(
            ['apt-get', '-qq', 'install', '-y', 'zstd'],
            check=True
        )

    # Validate archive before extraction
    print("Checking archive integrity...")

    with open(os.devnull, 'w') as devnull:
        result = subprocess.run(
            ['tar', '--zstd', '-tf', str(ARCHIVE)],
            stdout=devnull,
            stderr=subprocess.PIPE,
            text=True
        )

    if result.returncode != 0:
        raise RuntimeError(
            "Archive appears incomplete or invalid.\n"
            "Check the saved file in Google Drive.\n"
            f"{result.stderr[-1000:]}"
        )

    print("Archive verified.")

    # Extract only when not already marked complete
    print("Extracting to Google Drive...")

    subprocess.run(
        [
            'tar', '--zstd',
            '-xf', str(ARCHIVE),
            '-C', str(DATA_ROOT)
        ],
        check=True
    )

    if not dataset_present():
        raise RuntimeError(
            "Extraction finished but COCO annotations or "
            "images were not found. Inspect the archive structure."
        )

    # Only mark complete after successful extraction
    READY_MARKER.write_text(
        "M18K V2 extraction completed successfully.\n",
        encoding='utf-8'
    )

    print("Extraction completed successfully!")


# 6. Final verification
json_files = find_coco_jsons(DATA_ROOT)

print("\n" + "=" * 55)
print("M18K V2 DATASET STATUS")
print("=" * 55)

print("Dataset path:", DATA_ROOT)
print("Archive path:", ARCHIVE)
print("COCO annotation files:", len(json_files))

for p in json_files[:20]:
    print(" -", p.relative_to(DATA_ROOT))

if not json_files:
    raise FileNotFoundError(
        "No COCO annotation files found in extracted V2 dataset."
    )

print("\nDataset is ready for the next notebook cell.")

In [ ]:
print("Dataset:", DATA_ROOT)

for split in ['train', 'valid', 'test']:
    folder = DATA_ROOT / split
    print(
        f"{split}:",
        "FOUND" if folder.exists() else "MISSING"
    )

### 3 — Discover/validate annotations and images

This stage never silently renames categories or swaps test and validation. It verifies image existence, box geometry and polygon segmentation.

In [ ]:
import sys
import subprocess
import sqlite3
import json
import gc
from pathlib import Path
from collections import Counter
from PIL import Image

try:
    import ijson
except ImportError:
    subprocess.check_call([
        sys.executable, '-m', 'pip', '-q', 'install', 'ijson'
    ])
    import ijson

# Find annotation files
JSON_FILES = []
for split in ('train', 'valid', 'test'):
    split_dir = DATA_ROOT / split
    if split_dir.exists():
        JSON_FILES.extend(split_dir.glob('*.json'))

if not JSON_FILES:
    JSON_FILES = find_coco_jsons(DATA_ROOT)

JSON_FILES = sorted(set(JSON_FILES))

if not JSON_FILES:
    raise FileNotFoundError(
        f"No COCO annotation JSON files found in {DATA_ROOT}"
    )

print("Annotation files:")
for p in JSON_FILES:
    print(
        p.relative_to(DATA_ROOT),
        f"({p.stat().st_size / 2**20:.1f} MiB)"
    )


# Store annotations on local disk, not in RAM or Drive
DB_PATH = Path('/content/m18k_annotations.sqlite')

if DB_PATH.exists():
    DB_PATH.unlink()

db = sqlite3.connect(str(DB_PATH))

db.execute("""
    CREATE TABLE annotations (
        source TEXT,
        image_id INTEGER,
        payload TEXT
    )
""")

# A lazy annotation collection
class LazyAnnotations:
    def __init__(self, source, image_id):
        self.source = source
        self.image_id = image_id

    def __iter__(self):
        cursor = db.execute(
            """
            SELECT payload
            FROM annotations
            WHERE source = ? AND image_id = ?
            """,
            (self.source, self.image_id)
        )
        for (payload,) in cursor:
            yield json.loads(payload)

    def __len__(self):
        return db.execute(
            """
            SELECT COUNT(*)
            FROM annotations
            WHERE source = ? AND image_id = ?
            """,
            (self.source, self.image_id)
        ).fetchone()[0]


def split_name(json_path):
    parts = [p.lower() for p in json_path.parts]

    for alias, canonical in [
        ('train', 'train'),
        ('training', 'train'),
        ('valid', 'val'),
        ('validation', 'val'),
        ('val', 'val'),
        ('test', 'test')
    ]:
        if alias in parts:
            return canonical

    return None


def standard_name(name):
    n = name.lower().replace('_', ' ').replace('-', ' ').strip()

    if n in ('bb', 'baby bella', 'brown button', 'cremini'):
        return 'BB'

    if n in ('wb', 'white button', 'white mushroom'):
        return 'WB'

    return None


def locate_image(info, ann_file):
    name = str(info.get('file_name', '')).replace('\\', '/')
    if not name:
        return None

    rel = Path(name.lstrip('/'))

    choices = [
        ann_file.parent / rel,
        ann_file.parent / 'images' / rel.name,
        DATA_ROOT / rel,
        DATA_ROOT / 'images' / rel.name
    ]

    for base in [ann_file.parent.parent, ann_file.parent.parent.parent]:
        choices.extend([
            base / rel,
            base / 'images' / rel.name
        ])

    for candidate in choices:
        if candidate.is_file():
            return candidate.resolve()

    return None


sources = []
class_names = {}
total_annotations = 0

# Parse one record at a time instead of json.load()
for jp in JSON_FILES:
    print(f"\nProcessing: {jp.name} [{jp.parent.name}]")

    source_key = str(jp)

    # Categories: usually tiny
    with jp.open('rb') as f:
        cats = {
            int(c['id']): str(c['name']).strip()
            for c in ijson.items(
                f, 'categories.item', use_float=True
            )
        }

    if not cats:
        print("Skipping non-COCO JSON:", jp)
        continue

    cm = {
        cid: standard_name(name)
        for cid, name in cats.items()
    }

    if not {'BB', 'WB'}.issubset(set(cm.values())):
        raise ValueError(
            f"Unexpected categories in {jp}: {cats}. "
            "Inspect class names before proceeding."
        )

    print("Class mapping:", cm)

    # Stream annotations directly into SQLite
    count = 0
    batch = []

    with jp.open('rb') as f:
        for ann in ijson.items(
            f, 'annotations.item', use_float=True
        ):
            batch.append((
                source_key,
                int(ann['image_id']),
                json.dumps(ann, separators=(',', ':'))
            ))

            if len(batch) >= 500:
                db.executemany(
                    "INSERT INTO annotations VALUES (?, ?, ?)",
                    batch
                )
                batch.clear()

            count += 1

    if batch:
        db.executemany(
            "INSERT INTO annotations VALUES (?, ?, ?)",
            batch
        )

    db.commit()
    total_annotations += count

    print("Annotations indexed:", count)

    # Stream image metadata
    image_count = 0

    with jp.open('rb') as f:
        for im in ijson.items(
            f, 'images.item', use_float=True
        ):
            imid = int(im['id'])

            sources.append({
                'json_path': str(jp),
                'id': imid,
                'image_path': locate_image(im, jp),
                'width': int(im.get('width', 0)),
                'height': int(im.get('height', 0)),
                'annotations': LazyAnnotations(
                    source_key, imid
                ),
                'categories': cm,
                'original_split': split_name(jp)
            })

            image_count += 1

    print("Images discovered:", image_count)
    gc.collect()


# Index for fast per-image retrieval
print("\nBuilding SQLite index...")
db.execute("""
    CREATE INDEX idx_annotations_image
    ON annotations(source, image_id)
""")
db.commit()


# Validate file references without decoding images
missing = [
    s for s in sources
    if s['image_path'] is None
]

print("\nTotal image records:", len(sources))
print("Total annotations:", total_annotations)
print("Missing RGB images:", len(missing))

if missing:
    print("Example missing records:")
    for s in missing[:5]:
        print(s['json_path'], s['id'])

    raise FileNotFoundError(
        "Some images referenced by COCO JSON are missing."
    )

if not sources:
    raise RuntimeError("No images discovered.")


# Check dimensions on a small sample
for s in sources[:min(8, len(sources))]:
    with Image.open(s['image_path']) as im:
        w, h = im.size

    if s['width'] and s['height']:
        if (s['width'], s['height']) != (w, h):
            raise ValueError(
                f"Image dimension mismatch: {s['image_path']}"
            )

    s['width'], s['height'] = w, h


# Fill any missing image dimensions
for s in sources:
    if s['width'] <= 0 or s['height'] <= 0:
        with Image.open(s['image_path']) as im:
            s['width'], s['height'] = im.size


print("\n" + "=" * 50)
print("DATASET DISCOVERY COMPLETE")
print("=" * 50)
print("Images:", len(sources))
print("Annotations:", total_annotations)
print("SQLite index:", DB_PATH)
print("Dataset:", DATA_ROOT)
print("Proceed to split audit and EDA.")

In [ ]:
from pathlib import Path
from collections import defaultdict
import ijson

print("DATA ROOT:", DATA_ROOT)
print("\n===== FOLDER INSPECTION =====")

for split in ["train", "valid", "test"]:
    folder = DATA_ROOT / split
    print(f"\n{split}/")
    if folder.exists():
        for p in list(folder.iterdir())[:12]:
            print(" ", p.name, "(DIR)" if p.is_dir() else "(FILE)")
    else:
        print("  FOLDER MISSING")

# Build image lookup indexes
extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

by_name = defaultdict(list)
by_stem = defaultdict(list)
images_by_split = defaultdict(int)

print("\n===== INDEXING ACTUAL IMAGE FILES =====")

for p in DATA_ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in extensions:
        by_name[p.name.lower()].append(p)
        by_stem[p.stem.lower()].append(p)

        parts = p.relative_to(DATA_ROOT).parts
        if parts:
            images_by_split[parts[0]] += 1

print("Actual image files found:", sum(images_by_split.values()))
print("Images by top-level folder:", dict(images_by_split))

print("\nActual filename examples:")
for paths in list(by_name.values())[:5]:
    print(" ", paths[0].relative_to(DATA_ROOT))


# Match COCO image metadata to existing sources
source_lookup = {
    (s["json_path"], int(s["id"])): s
    for s in sources
}

fixed = 0
unresolved = []

print("\n===== REPAIRING IMAGE PATHS =====")

for jp in JSON_FILES:
    jp = Path(jp)
    expected_split = jp.parent.name.lower()

    with jp.open("rb") as f:
        for im in ijson.items(f, "images.item"):
            image_id = int(im["id"])
            filename = str(im.get("file_name", "")).replace("\\", "/")
            basename = Path(filename).name

            key = (str(jp), image_id)
            s = source_lookup.get(key)

            if s is None:
                continue

            # Find matching files by filename
            candidates = by_name.get(basename.lower(), [])

            # Prefer files inside the corresponding split
            split_matches = [
                p for p in candidates
                if expected_split in [
                    x.lower() for x in p.relative_to(DATA_ROOT).parts
                ]
            ]

            if len(split_matches) == 1:
                match = split_matches[0]
            elif len(candidates) == 1:
                match = candidates[0]
            else:
                # Try matching filename without extension
                stem_matches = by_stem.get(
                    Path(basename).stem.lower(), []
                )

                split_stem_matches = [
                    p for p in stem_matches
                    if expected_split in [
                        x.lower() for x in p.relative_to(DATA_ROOT).parts
                    ]
                ]

                match = (
                    split_stem_matches[0]
                    if len(split_stem_matches) == 1
                    else stem_matches[0]
                    if len(stem_matches) == 1
                    else None
                )

            if match is not None:
                s["image_path"] = match
                fixed += 1
            else:
                unresolved.append(
                    (expected_split, filename)
                )

print("\n===== RESULTS =====")
print("Paths repaired:", fixed)
print("Still unresolved:", len(unresolved))

if unresolved:
    print("\nCOCO filenames that could not be matched:")
    for item in unresolved[:10]:
        print(" ", item)

if fixed == len(sources):
    print("\nSUCCESS! All image paths are connected.")
    print("You can continue to the next EDA cell.")
else:
    print("\nSome image files could not be located.")
    print("Check the examples above before proceeding.")

### 4 — Split audit and creation

In [ ]:
from collections import Counter
splits_present = {s['original_split'] for s in sources}
if splits_present == {'train','val','test'}:
    for s in sources: s['split']=s['original_split']
    split_source='Dataset-provided COCO split'
else:
    # Any ambiguous/missing split => use a new fixed seed across all images; never swap existing val/test by accident.
    rng = np.random.default_rng(SEED)
    order = list(rng.permutation(len(sources)))
    n = len(order)
    n_train = int(round(n*.70)); n_val=int(round(n*.10))
    lookup = {idx: ('train' if k<n_train else 'val' if k<n_train+n_val else 'test') for k,idx in enumerate(order)}
    for i,s in enumerate(sources): s['split']=lookup[i]
    split_source='New seeded random 70/10/20 image split (not paper split)'
    print('WARNING: generated random image split; investigate frame/scene leakage.')

print(split_source)
print('Image counts:', Counter(s['split'] for s in sources))

# Verify exact byte-identical images do not cross splits. Does not detect near duplicates.
def file_sha256(path):
    dig=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            dig.update(chunk)
    return dig.hexdigest()
CHECK_EXACT_DUPLICATES = False
if CHECK_EXACT_DUPLICATES:
    by_hash=defaultdict(list)
    for s in sources:
        by_hash[file_sha256(s['image_path'])].append(s['split'])
    leaking = {k:v for k,v in by_hash.items() if len(set(v))>1}
    print('Exact image-content hashes occurring across splits:',len(leaking))
    if leaking:
        raise RuntimeError('Exact duplicate image leakage across splits. Resolve before training.')

### 5 — EDA and annotation quality-control

Figures are saved to `WORK_ROOT/eda/`; annotation statistics are also saved as CSVs.

In [ ]:
EDA_DIR = WORK_ROOT/'eda'
EDA_DIR.mkdir(exist_ok=True,parents=True)
records=[]; invalid=[]; class_instance_counts=Counter(); image_counts=Counter(); empty_masks=0
for s in sources:
    label_per_image=Counter()
    w,h=s['width'],s['height']
    for ann in s['annotations']:
        label=s['categories'].get(int(ann['category_id']))
        if label not in ('BB','WB'): continue
        box=ann.get('bbox',[])   # xywh
        if len(box)!=4 or not all(np.isfinite(box)):
            invalid.append(('invalid_bbox',s['json_path'],s['id'],ann.get('id')));continue
        x,y,bw,bh=[float(v) for v in box]
        if bw<=0 or bh<=0 or x<0 or y<0 or x+bw>w+2 or y+bh>h+2:
            invalid.append(('bbox_outside',s['json_path'],s['id'],ann.get('id')))
        seg=ann.get('segmentation')
        if not seg: empty_masks+=1
        records.append({'split':s['split'],'class':label,'bbox_width':bw,'bbox_height':bh,
                        'bbox_area_px':bw*bh,'bbox_area_fraction':bw*bh/(max(w*h,1)),
                        'annotation_area_px':ann.get('area',np.nan),
                        'image_id':str(s['json_path'])+'::'+str(s['id']),
                        'img_w':w,'img_h':h,'has_segmentation':bool(seg)})
        label_per_image[label]+=1
        class_instance_counts[label]+=1
    image_counts[(s['split'],'images')]+=1
    image_counts[(s['split'],'instances')]+=sum(label_per_image.values())

eda=pd.DataFrame.from_records(records)
if eda.empty: raise ValueError('No BB or WB annotations found')
images_summary=pd.DataFrame([{'split':sp,'images':image_counts[(sp,'images')],
                              'instances':image_counts[(sp,'instances')]} for sp in ('train','val','test')])
classes_summary=pd.crosstab(eda['split'],eda['class']).reindex(index=['train','val','test'],columns=['BB','WB'],fill_value=0)
print('Image summary:');display(images_summary)
print('BB/WB instance distribution:');display(classes_summary)
print('Malformed/out-of-range boxes:',len(invalid),'| Missing segmentation fields:',empty_masks)
def depth_file_for_image(image_path):
    ip = Path(image_path)
    # Original M18K: split/images/img.jpg -> split/depth/img.npy
    parts=list(ip.parts)
    for i in range(len(parts)-1,-1,-1):
        if parts[i].lower() == 'images':
            for folder in ('depth','depths'):
                alt=parts.copy();alt[i]=folder
                for ext in ('.npy','.png','.tiff','.tif'):
                    candidate=Path(*alt).with_suffix(ext)
                    if candidate.is_file():return candidate
    return None

paired_depth={s['split']:0 for s in sources}
for s in sources:
    s['depth_path']=depth_file_for_image(s['image_path'])
    if s['depth_path'] is not None: paired_depth[s['split']]+=1
print('Found RGB-depth pairs per split:',dict(paired_depth))
pd.DataFrame([{'split':sp,'rgb':int(sum(s['split']==sp for s in sources)),
               'rgb_with_depth':int(paired_depth.get(sp,0))} for sp in ('train','val','test')]).to_csv(
                  EDA_DIR/'rgb_depth_pairing.csv',index=False)

images_summary.to_csv(EDA_DIR/'images_by_split.csv',index=False)
classes_summary.to_csv(EDA_DIR/'instances_by_split.csv')
eda.to_csv(EDA_DIR/'annotations_summary.csv',index=False)
pd.DataFrame(invalid,columns=['problem','json','image_id','annotation_id']).to_csv(EDA_DIR/'invalid_annotations.csv',index=False)

fig, axs=plt.subplots(1,3,figsize=(15,4))
images_summary.plot.bar(x='split',y='images',ax=axs[0],legend=False,title='Images per split',rot=0)
classes_summary.plot.bar(ax=axs[1],title='Mushroom instances by class',rot=0)
for name in ('BB','WB'):
    a=eda.loc[eda['class']==name,'bbox_area_fraction'].dropna()
    if len(a): axs[2].hist(a,bins=40,alpha=.6,label=name)
axs[2].set_title('Bounding-box fraction of image');axs[2].set_xlabel('Box area / image area');axs[2].legend()
plt.tight_layout();plt.savefig(EDA_DIR/'eda_overview.png',dpi=160);plt.show()

counts=eda.groupby('image_id').size()
fig, ax=plt.subplots(figsize=(8,3))
ax.hist(counts,bins=30);ax.set_title('Number of annotated mushrooms per image')
ax.set_xlabel('Instances/image');ax.set_ylabel('Number of images')
fig.tight_layout();fig.savefig(EDA_DIR/'instances_per_image.png',dpi=160);plt.show()

In [ ]:
def show_example(s, ax):
    with Image.open(s['image_path']) as im:
        img=np.array(im.convert('RGB'))
    ax.imshow(img)
    colors={'BB':'deepskyblue','WB':'orange'}
    shown=0
    for a in s['annotations']:
        name=s['categories'].get(int(a['category_id']))
        if name not in colors: continue
        box=a.get('bbox',[])
        if len(box)==4:
            x,y,w,h=box
            ax.add_patch(plt.Rectangle((x,y),w,h,fill=False,edgecolor=colors[name],linewidth=.6))
        seg=a.get('segmentation')
        if isinstance(seg,list):
            for part in seg:
                if len(part)>=6:
                    p=np.array(part,dtype=float).reshape(-1,2)
                    ax.plot(np.r_[p[:,0],p[0,0]],np.r_[p[:,1],p[0,1]],color=colors[name],linewidth=.6)
        shown+=1
    ax.set_title(f'{s["split"]} | {shown} annotated instances');ax.axis('off')

rng=np.random.default_rng(SEED)
fig,axes=plt.subplots(2,3,figsize=(16,9));axes=axes.ravel()
for ax,idx in zip(axes,rng.choice(len(sources),size=min(len(sources),len(axes)),replace=False)):
    show_example(sources[int(idx)],ax)
for ax in axes[min(len(sources),len(axes)):]:ax.axis('off')
plt.tight_layout();plt.savefig(EDA_DIR/'sample_annotations.png',dpi=160);plt.show()

available_depth=[s for s in sources if s.get('depth_path') is not None]
if available_depth:
    pick=available_depth[0]
    with Image.open(pick['image_path']) as im: rgb=np.array(im.convert('RGB'))
    dp=pick['depth_path']
    depth=np.load(dp) if dp.suffix.lower()=='.npy' else np.array(Image.open(dp))
    depth=np.squeeze(depth)
    if depth.ndim==2:
        fig,axs=plt.subplots(1,2,figsize=(12,5))
        axs[0].imshow(rgb);axs[0].set_title('RGB image');axs[0].axis('off')
        dvalid=depth[np.isfinite(depth) & (depth>0)]
        axs[1].imshow(depth,cmap='viridis',vmin=np.percentile(dvalid,1) if len(dvalid) else None,
                      vmax=np.percentile(dvalid,99) if len(dvalid) else None)
        axs[1].set_title(f'Depth map — valid pixels: {np.mean(np.isfinite(depth)&(depth>0)):.1%}')
        axs[1].axis('off')
        plt.tight_layout();plt.savefig(EDA_DIR/'sample_rgb_depth.png',dpi=160);plt.show()
    else: print('Depth dimensions need investigation:', dp, depth.shape)
else:
    print('No depth maps paired — do not claim RGB-D EDA is complete.')

### 6 — Convert COCO instance masks to Ultralytics segmentation format

**Conversion policy:** COCO polygons are used directly when possible. For multi-part masks, only the largest polygon component is retained because a YOLO segmentation label encodes one polygon per instance. RLE masks are decoded into external contours. The notebook reports all approximations. Any missing segmentation is skipped, **never replaced with fake box masks**.

Image files are symlinked when possible to avoid doubling disk usage. The output `data.yaml` explicitly points `val → images/val` and `test → images/test`.

In [ ]:
import yaml
YOLO_ROOT=WORK_ROOT/'yolo_v2'
for split in ('train','val','test'):
    (YOLO_ROOT/'images'/split).mkdir(parents=True,exist_ok=True)
    (YOLO_ROOT/'labels'/split).mkdir(parents=True,exist_ok=True)

conv_stats=Counter()
def polygon_area(points):
    p=np.array(points,dtype=float).reshape(-1,2)
    if len(p)<3: return 0
    return abs(np.dot(p[:,0],np.roll(p[:,1],1))-np.dot(p[:,1],np.roll(p[:,0],1)))/2

def segmentation_to_polygon(annotation,h,w):
    seg=annotation.get('segmentation')
    if not seg: return None
    if isinstance(seg,list):
        valid=[]
        for poly in seg:
            if len(poly)>=6 and len(poly)%2==0:
                a=polygon_area(poly)
                if a>1: valid.append((a,poly))
        if not valid: return None
        if len(valid)>1:conv_stats['multipart_instances_kept_largest']+=1
        return max(valid,key=lambda x:x[0])[1]
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):
            rle=coco_mask.frPyObjects(seg,h,w)
        else: rle=seg
        mask=coco_mask.decode(rle)
        if mask.ndim==3:mask=mask[:,:,0]
        contours,_=cv2.findContours(mask.astype('uint8'),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        contours=[c for c in contours if len(c)>=3 and cv2.contourArea(c)>1]
        if not contours:return None
        if len(contours)>1: conv_stats['multipart_instances_kept_largest']+=1
        return max(contours,key=cv2.contourArea).reshape(-1).tolist()
    return None

def yolo_seg_line(annotation,s):
    name=s['categories'].get(int(annotation['category_id']))
    if name not in ('BB','WB'):
        conv_stats['skipped_other_class']+=1;return None
    if int(annotation.get('iscrowd',0)):
        conv_stats['skipped_crowd']+=1;return None
    poly=segmentation_to_polygon(annotation,int(s['height']),int(s['width']))
    if poly is None or len(poly)<6:
        conv_stats['skipped_missing_or_bad_mask']+=1;return None
    xy=np.array(poly,dtype=np.float64).reshape(-1,2)
    if not np.isfinite(xy).all():
        conv_stats['skipped_nonfinite_polygon']+=1;return None
    xy[:,0]=np.clip(xy[:,0]/s['width'],0,1)
    xy[:,1]=np.clip(xy[:,1]/s['height'],0,1)
    class_id=0 if name=='BB' else 1
    conv_stats['written_instances']+=1
    return str(class_id)+' '+' '.join(f'{p:.6f}' for p in xy.flatten())

manifest=[]
for s in sources:
    key=f'{s["json_path"]}::{s["id"]}::{s["image_path"]}'
    stem=hashlib.sha1(key.encode()).hexdigest()[:15]
    im_dst=YOLO_ROOT/'images'/s['split']/(stem+s['image_path'].suffix.lower())
    label_dst=YOLO_ROOT/'labels'/s['split']/(stem+'.txt')
    if not im_dst.exists():
        try: im_dst.symlink_to(s['image_path'])
        except OSError: shutil.copy2(s['image_path'],im_dst)
    lines=[]
    for a in s['annotations']:
        line=yolo_seg_line(a,s)
        if line:lines.append(line)
    label_dst.write_text('\n'.join(lines)+('\n' if lines else ''),encoding='utf-8')
    manifest.append({'split':s['split'],'original_image':str(s['image_path']),
                     'image':str(im_dst),'label':str(label_dst),'instances':len(lines)})

cfg={'path':str(YOLO_ROOT.resolve()),'train':'images/train','val':'images/val',
     'test':'images/test','names':{0:'BB',1:'WB'}}
with open(YOLO_ROOT/'data.yaml','w',encoding='utf-8') as f: yaml.safe_dump(cfg,f,sort_keys=False)
pd.DataFrame(manifest).to_csv(WORK_ROOT/'image_manifest.csv',index=False)
print('Created:',YOLO_ROOT/'data.yaml')
print('Conversion statistics:',dict(conv_stats))
print('Split distribution:',Counter(x['split'] for x in manifest))
if conv_stats['written_instances']==0: raise RuntimeError('No masks converted — do not train.')
if len(manifest) < 100: print('WARNING: very small dataset, check extraction completeness.')

### 7 — Train the 640-pixel pilot (GPU recommended)

This is a *first reproducible experiment*, not a verified duplicate of the published full training setup. The paper specifies AdamW with initial LR `1e-4`, cosine annealing, crop/rotation/translation augmentation. We approximate the compatible YOLO augmentations; **cropping is not matched exactly**. We disable Ultralytics-specific mosaic/mixup and color augmentations for clearer comparability.

For a stronger experiment after the first run, repeat with more epochs and `IMAGE_SIZE=1280`, keeping the final test set untouched during model selection.

In [ ]:
from ultralytics import YOLO
import torch
if RUN_TRAIN and (torch.cuda.is_available() or ALLOW_CPU_TRAIN):
    model=YOLO('yolov8n-seg.pt')
    device=0 if torch.cuda.is_available() else 'cpu'
    RUN_NAME=f'yolov8n_seg_v2_s{IMAGE_SIZE}_e{EPOCHS}_seed{SEED}'
    _train=model.train(data=str(YOLO_ROOT/'data.yaml'),
                       epochs=EPOCHS,imgsz=IMAGE_SIZE,batch=BATCH,
                       optimizer='AdamW',lr0=1e-4,cos_lr=True,
                       degrees=15,translate=0.10,scale=0.20,
                       hsv_h=0.0,hsv_s=0.0,hsv_v=0.0,
                       fliplr=0.0,flipud=0.0,mosaic=0.0,mixup=0.0,copy_paste=0.0,
                       seed=SEED,deterministic=True,workers=2,amp=(device!= 'cpu'),
                       project=str(WORK_ROOT/'runs'),name=RUN_NAME,exist_ok=True,
                       device=device,plots=True)
    BEST=WORK_ROOT/'runs'/RUN_NAME/'weights'/'best.pt'
    print('Trained checkpoint:',BEST,'Exists:',BEST.is_file())
else:
    BEST=None
    print('Training skipped: set GPU runtime in Colab (or explicitly ALLOW_CPU_TRAIN=True).')

### 8 — Evaluate the held-out test set **once** and save metrics

Do not tune hyperparameters based on test results. Use the validation split to select the checkpoint. The paper reports AP; compare it to Ultralytics `metrics.seg.map` (AP@50:95) **with the explicit caveat that the V2 data/split may differ**.

In [ ]:
PAPER_REFERENCE={'YOLOv8-N-seg-640':{'F1':0.748,'AP':0.712,'AR':0.788},
                 'YOLOv8-N-seg-1280':{'F1':0.859,'AP':0.846,'AR':0.874}}
if BEST is not None and BEST.is_file():
    best_model=YOLO(str(BEST))
    TEST_METRICS=best_model.val(data=str(YOLO_ROOT/'data.yaml'),split='test',
                                imgsz=IMAGE_SIZE,batch=BATCH,device=0 if torch.cuda.is_available() else 'cpu',
                                plots=True,project=str(WORK_ROOT/'runs'),name='heldout_test',exist_ok=True)
    measurement={
        'dataset':'M18K V2', 'split_source':split_source,
        'test_images':int(sum(s['split']=='test' for s in sources)),
        'model':'YOLOv8n-seg','imgsz':IMAGE_SIZE,'epochs':EPOCHS,'seed':SEED,
        'box_precision':float(TEST_METRICS.box.mp),'box_recall':float(TEST_METRICS.box.mr),
        'box_mAP50':float(TEST_METRICS.box.map50),'box_mAP50_95':float(TEST_METRICS.box.map),
        'mask_precision':float(TEST_METRICS.seg.mp),'mask_recall':float(TEST_METRICS.seg.mr),
        'mask_mAP50':float(TEST_METRICS.seg.map50),'mask_mAP50_95':float(TEST_METRICS.seg.map),
        'paper_YOLOv8N_640_mask_AP':PAPER_REFERENCE['YOLOv8-N-seg-640']['AP'],
        'comparable_to_original_paper':False,
        'reason':'V2 version and test identities/protocol not verified identical to paper',
    }
    out=WORK_ROOT/'heldout_test_metrics.json'
    out.write_text(json.dumps(measurement,indent=2),encoding='utf-8')
    display(pd.DataFrame([measurement]).T.rename(columns={0:'Value'}))
    print('Saved',out)
else:
    print('No trained checkpoint in this runtime, so NO new model-performance numbers are claimed.')

### 9 — Export EDA results

In [ ]:
import zipfile
summary={
 'reference':'Zakeri et al. Computers 2025 14(5):199, Table 2',
 'data_version':'V2','claimed_repository_size':4652,'discovered_images':len(sources),
 'split_origin':split_source,'split_counts':dict(Counter(s['split'] for s in sources)),
 'instances':dict(class_instance_counts), 'depth_pairs_by_split':dict(paired_depth), 'invalid_bbox_count':len(invalid),
 'missing_mask_count':empty_masks,'mask_conversion':dict(conv_stats),
 'trained':bool(BEST is not None and BEST.is_file())
}
(WORK_ROOT/'day1_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
report=['# M18K V2 — reproduction experiment report',
        f'- Data/version: V2; {len(sources)} discovered images',
        f'- Split: {split_source}',
        f'- Image counts: {dict(Counter(s["split"] for s in sources))}',
        f'- Instance counts: {dict(class_instance_counts)}',
        f'- RGB/depth pairs: {dict(paired_depth)}',
        f'- Bad bboxes: {len(invalid)}; missing masks: {empty_masks}',
        f'- COCO -> YOLO conversion: {dict(conv_stats)}',
        f'- Training completed: {bool(BEST is not None and BEST.is_file())}',
        '- Paper YOLOv8-N segmentation AP: 0.712 (640), 0.846 (1280)',
        '- This V2 pilot is NOT directly comparable without matching the original dataset/test identities.',
        '- **Next:** inspect missing depth pairs; verify scene-based split leakage; increase training horizon; evaluate 1280; add 3D.']
if (WORK_ROOT/'heldout_test_metrics.json').exists():
    met=json.loads((WORK_ROOT/'heldout_test_metrics.json').read_text())
    report.append(f'- Our test mask mAP50:95: {met["mask_mAP50_95"]:.4f} (different experiment)')
(WORK_ROOT/'day1_report.md').write_text('\n'.join(report)+'\n',encoding='utf-8')

EXPORT=ROOT/'M18K_V2_day1_results.zip'
with zipfile.ZipFile(EXPORT,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in [WORK_ROOT/'day1_report.md',WORK_ROOT/'day1_summary.json',WORK_ROOT/'heldout_test_metrics.json',WORK_ROOT/'image_manifest.csv',
              YOLO_ROOT/'data.yaml'] + list(EDA_DIR.glob('*')):
        if p.is_file(): z.write(p,arcname=p.relative_to(WORK_ROOT))
print('Exported:',EXPORT)
print('Files',len(zipfile.ZipFile(EXPORT).namelist()))



**Stage 1 (DS/EDA):** correct V2 archive extraction; COCO/images/depth audit; split integrity; class/size/density plots; sample masks; a Git-ready EDA report.

**Stage 2 (pilot):** YOLOv8n-seg training finishes; a `best.pt` exists; held-out test mask/box metrics are saved, without test-set tuning.

**Stage 3 (strict paper reproduction):** recover **exact** 2,000-image benchmark data and original split, align training and AP evaluation with the paper, and run 640/1280 resolution experiments; only then report agreement/deviation from Table 2 as reproduction findings.

### Known limitations
- Official V2 is described as 4,652 images, while the paper specifies 2,000 images. Identity of original benchmark split is **not established**.
- If the V2 archive contains only one COCO file, the generated random image split can leak visually similar camera sequences. Near-duplicate/scene verification is still necessary.
- Largest-polygon conversion approximates multipart segmentation instances. Its impact must be audited.
- 10 epochs are a practical first run, **not** paper-matched full convergence.
- RGB-D pose estimation is future experimental work; this notebook does **not** pretend to evaluate 3D without annotations and pose code.